# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** Foam Finger

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

expected_rows = sum(len(o['lines']) for o in orders_json)
expected_qty = sum(l['qty'] for o in orders_json for l in o['lines'])

assert len(lines) == expected_rows == 4, f"row count: {len(lines)}"
assert lines['qty'].sum() == expected_qty == 7, f"qty total: {lines['qty'].sum()}"

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors = pd.DataFrame(vendors_json)

joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True, validate='many_to_one')

assert len(joined) == len(lines)

unmatched = joined[joined['_merge'] == 'left_only']
report = unmatched.groupby('vendor_id')['qty'].sum()

print(joined['_merge'].value_counts())
print("Unmatched vendor and units:", report.to_dict() or "none")

_merge
both          4
left_only     0
right_only    0
Name: count, dtype: int64
Unmatched vendor and units: none


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
joined['vendor_name'] = joined['name'].fillna('UNMATCHED (' + joined['vendor_id'] + ')')

qty_by_vendor = (
    joined.groupby('vendor_name', as_index=False)['qty'].sum()
          .sort_values('qty', ascending=False, ignore_index=True)
)

assert qty_by_vendor['qty'].sum() == lines['qty'].sum() == 7

#Unmatched vendors stay in the table as their own labelled row, so the totals still add up to the 7 units from Q1.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:

zone_qty = joined.groupby('zone')['qty'].sum().sort_values(ascending=False)

grand_total = joined['qty'].sum()
zoned_total = zone_qty.sum()
no_zone = joined[joined['zone'].isna()].groupby('vendor_id')['qty'].sum()

assert zoned_total + no_zone.sum() == grand_total == 7   # every unit is accounted for

print(zone_qty)
print(f"Top zone: {zone_qty.idxmax()} ({zone_qty.max()} units)")
print(f"Zoned total: {zoned_total} of {grand_total}, Difference: {grand_total - zoned_total}, {no_zone.to_dict() or 'no unmatched vendors'}")

# Rows with no zone (unmatched vendors) are deliberately left out of the zone ranking, since they can't be attributed to a zone, and are reported separately so the gap to the grand total is accounted for.

zone
A    7
Name: qty, dtype: int64
Top zone: A (7 units)
Zoned total: 7 of 7, Difference: 0, no unmatched vendors


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# Order 5 has no `lines` key, which makes json_normalize raise KeyError, so give those orders an empty list
patched = [{**o, 'lines': o.get('lines') or []} for o in ragged_json]
ragged = pd.json_normalize(patched, record_path='lines', meta=['order', 'vendor_id'])
# Guarantee the columns exist even if no line in the batch carries a qty.
ragged = ragged.reindex(columns=['item', 'qty', 'order', 'vendor_id'])
ragged['qty'] = ragged['qty'].astype('Int64')

# Orders with no lines produce no rows, so report them instead of losing them.
no_lines = [o['order'] for o in ragged_json if not o.get('lines')]

assert len(ragged) == sum(len(o.get('lines') or []) for o in ragged_json) == 2
assert ragged['qty'].isna().sum() == 1      # Fries: quantity unknown
assert (ragged['qty'] == 0).sum() == 0      # nothing was coerced to zero
assert ragged['qty'].sum() == 2      # known units only; <NA> is skipped

print("Orders with no lines:", no_lines)
ragged

Orders with no lines: [5]


,item,qty,order,vendor_id
0,Soda,2,4,V-01
1,Fries,<NA>,6,V-01


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** Quantity belongs to a line not an order, so one row per line item gives each row exactly one item and qty. Questions like "how many Sodas were sold?" or "units per vendor or zone" become a single `groupby(...)['qty'].sum()`. With one row per order, the lines sit in a list inside a cell that pandas can't sum or group if itsn not firstly exploded.

**b)** Zero means we know none were ordered, but missing means the line exists but we don't know how many. Conflating them makes totals look complete when they are not, and drags averages down (mean units per line is 2 excluding the missing value, 1 counting it as zero), and makes the bad record indistinguishable from real zeros.